In [1]:
# دي بتنادي على المكتبات اللي بتخلي بايثون يكلم تيراداتا
import teradataml as tdml
from teradataml import create_context

# افتح الاتصال (هنا Kernel هيربط فعلياً بالداتا بيز)
con = create_context(host="marketingproject-f0wngnec96lhx7am.env.clearscape.teradata.com", username="demo_user", password="Mahmoud_yavus1")

print("Connection Successful!")

Connection Successful!


In [ ]:
from teradataml import execute_sql

# 1. إنشاء جدول العملاء المحتملين
execute_sql("""
CREATE TABLE demo_user.BRZ_CRM_LEADS (
    lead_id VARCHAR(100),
    created_at VARCHAR(100),
    updated_at VARCHAR(100),
    email_hash VARCHAR(255),
    phone_hash VARCHAR(100),
    lead_source VARCHAR(100),
    status_current VARCHAR(50),
    owner_id VARCHAR(50)
) PRIMARY INDEX (lead_id);
""")

# 2. إنشاء جدول الفرص البيعية
execute_sql("""
CREATE TABLE demo_user.BRZ_CRM_OPPORTUNITIES (
    opportunity_id VARCHAR(100),
    lead_id VARCHAR(100),
    created_at VARCHAR(100),
    updated_at VARCHAR(100),
    stage VARCHAR(50),
    amount VARCHAR(50),
    currency VARCHAR(20),
    close_date VARCHAR(100)
) PRIMARY INDEX (opportunity_id);
""")

# 3. إنشاء جدول صرف الإعلانات
execute_sql("""
CREATE TABLE demo_user.BRZ_ADS_SPEND (
    spend_date VARCHAR(100),
    source_system VARCHAR(100),
    account_id VARCHAR(100),
    campaign_id VARCHAR(100),
    adset_id VARCHAR(100),
    ad_id VARCHAR(100),
    currency VARCHAR(20),
    impressions VARCHAR(50),
    clicks VARCHAR(50),
    spend_amount VARCHAR(50)
) PRIMARY INDEX (spend_date, campaign_id);
""")

# 4. إنشاء جدول أحداث الويب
execute_sql("""
CREATE TABLE demo_user.BRZ_WEB_EVENTS (
    event_id VARCHAR(100),
    event_ts VARCHAR(100),
    event_name VARCHAR(100),
    session_id VARCHAR(100),
    cookie_id VARCHAR(100),
    user_id VARCHAR(100),
    email_hash VARCHAR(255),
    utm_source VARCHAR(100),
    utm_medium VARCHAR(100),
    utm_campaign VARCHAR(100),
    page_url VARCHAR(255),
    device_type VARCHAR(50),
    geo_country VARCHAR(50),
    geo_city VARCHAR(50)
) PRIMARY INDEX (event_id);
""")

print("Bronze Tables Created Successfully!")

In [ ]:
# هنستخدم الـ context اللي إنت لسه فاتحه عشان ننفذ query تجيب الجداول
from teradataml import execute_sql
import pandas as pd

# query بتسأل الداتا بيز: إيه الجداول اللي تبع اليوزر بتاعي؟
query = "SELECT TableName FROM DBC.TablesV WHERE DatabaseName = 'demo_user' AND TableKind = 'T';"

# تنفيذ الـ query وعرض النتائج
try:
    results = execute_sql(query)
    # تحويل النتيجة لـ list وعرضها
    tables = [row[0] for row in results]
    print("Tables found in demo_user:")
    for table in tables:
        print(f"- {table}")
except Exception as e:
    print(f"Error: {e}")

In [5]:
import teradatasql

# --- بيانات الاتصال ---
host = "marketingproject-f0wngnec96lhx7am.env.clearscape.teradata.com"  # تأكد إن ده الـ IP الصح بتاع السيرفر عندك
username = "demo_user"           # أو اليوزر اللي إنت شغال بيه
password = "Mahmoud_yavus1"          # الباسورد بتاعك

with teradatasql.connect(host=host, user=username, password=password) as connect:
    cursor = connect.cursor()
    print("Connected to Teradata Successfully!")

Connected to Teradata Successfully!


In [5]:


# --- قائمة الجداول ---
tables = [
    "BRZ_ADS_SPEND", 
    "BRZ_CRM_OPPORTUNITIES", 
    "BRZ_WEB_EVENTS", 
    "BRZ_CRM_LEADS"
]

print("--- Starting Connection and Counting ---")

try:
    # 1. فتح الاتصال
    with teradatasql.connect(host=host, user=username, password=password) as connect:
        cursor = connect.cursor()
        print("Connected to Teradata Successfully!")
        
        print("\n--- Bronze Tables Data Counts ---")
        
        # 2. اللف على الجداول وعد الصفوف
        for table in tables:
            try:
                cursor.execute(f"SELECT COUNT(*) FROM {table}")
                row_count = cursor.fetchone()[0]
                print(f"Table {table}: {row_count} rows found.")
            except Exception as table_error:
                print(f"Error checking table {table}: {table_error}")
                
        print("\n-----------------------------------")

except Exception as conn_error:
    print(f"Failed to connect or execute: {conn_error}")

--- Starting Connection and Counting ---
Connected to Teradata Successfully!

--- Bronze Tables Data Counts ---
Table BRZ_ADS_SPEND: 60 rows found.
Table BRZ_CRM_OPPORTUNITIES: 224 rows found.
Table BRZ_WEB_EVENTS: 2102 rows found.
Table BRZ_CRM_LEADS: 431 rows found.

-----------------------------------


In [ ]:
import teradatasql
from teradataml import execute_sql

execute_sql("""
CREATE TABLE SLV_SALES_FUNNEL (
    lead_id INTEGER,
    opportunity_id INTEGER,
    lead_created_at TIMESTAMP,
    opportunity_created_at TIMESTAMP,
    stage VARCHAR(50),
    amount DECIMAL(18,2),
    conversion_days INTEGER
) PRIMARY INDEX (lead_id);
""")
print("Silver table SLV_SALES_FUNNEL created!")

In [ ]:
import teradatasql
from teradataml import execute_sql

execute_sql("""
    CREATE TABLE SLV_MARKETING_PERFORMANCE (
        campaign_id VARCHAR(50),
        total_spend DECIMAL(18,2),
        total_clicks INTEGER,
        total_web_events INTEGER
    ) PRIMARY INDEX (campaign_id);
    """)

    # الجدول الثاني: رحلة العميل (ربط الزيارات بالـ Leads)
execute_sql("""
    CREATE TABLE SLV_CUSTOMER_JOURNEY (
        fingerprint VARCHAR(100),
        lead_id INTEGER,
        event_type VARCHAR(50),
        event_timestamp TIMESTAMP
    ) PRIMARY INDEX (fingerprint);
    """)
print("Silver tables created successfully!")

In [8]:
import pandas as pd
import teradatasql
from teradataml import execute_sql

host = "marketingproject-f0wngnec96lhx7am.env.clearscape.teradata.com"  # تأكد إن ده الـ IP الصح بتاع السيرفر عندك
user = "demo_user"           # أو اليوزر اللي إنت شغال بيه
password = "Mahmoud_yavus1"          # الباسورد بتاعك


try:
    connect = teradatasql.connect(host=host, user=user, password=password)
    cursor = connect.cursor()
    print("✅ Connection Established Successfully!")
except Exception as e:
    print(f"❌ Connection Failed: {e}")
    
# قائمة جداول الـ Silver Layer التي قمنا بإنشائها
silver_tables = [
    'SLV_SALES_FUNNEL', 
    'SLV_MARKETING_PERFORMANCE', 
    'SLV_CUSTOMER_JOURNEY'
]

print("--- 🛡️ Silver Layer Data Validation 🛡️ ---")
print("-" * 50)

# 1. فحص عدد الصفوف في كل جدول باستخدام الـ Cursor
for table in silver_tables:
    try:
        query = f"SELECT COUNT(*) FROM {table}"
        cursor.execute(query)
        count = cursor.fetchone()[0]
        
        # تحديد حالة الجدول بناءً على وجود بيانات
        status = "✅ Data Loaded" if count > 0 else "⚠️ Empty Table"
        print(f"Table: {table:<30} | Rows: {count:<6} | Status: {status}")
    except Exception as e:
        print(f"❌ Error checking {table}: {e}")

print("-" * 50)

# 2. عرض عينة من البيانات للتأكد من صحة الـ Join والـ Calculations
print("\n--- 🔍 Quick Sample: Sales Funnel (Leads to Opportunities) ---")
try:
    # جلب عينة تشمل الـ Leads التي تحولت لـ Opportunities فعلاً للتأكد من حساب الأيام
    # استخدمنا pd.read_sql عشان نعرض النتيجة كجدول DataFrame شيك
    sample_query = "SELECT TOP 5 * FROM SLV_SALES_FUNNEL WHERE conversion_days IS NOT NULL"
    sample_df = pd.read_sql(sample_query, connect)
    
    if not sample_df.empty:
        print(sample_df)
    else:
        print("No converted leads found in sample. Checking all rows...")
        print(pd.read_sql("SELECT TOP 5 * FROM SLV_SALES_FUNNEL", connect).head())
except Exception as e:
    print(f"Could not fetch sample: {e}")

✅ Connection Established Successfully!
--- 🛡️ Silver Layer Data Validation 🛡️ ---
--------------------------------------------------
Table: SLV_SALES_FUNNEL               | Rows: 253    | Status: ✅ Data Loaded
Table: SLV_MARKETING_PERFORMANCE      | Rows: 3      | Status: ✅ Data Loaded
Table: SLV_CUSTOMER_JOURNEY           | Rows: 862    | Status: ✅ Data Loaded
--------------------------------------------------

--- 🔍 Quick Sample: Sales Funnel (Leads to Opportunities) ---
   lead_id  opportunity_id lead_created_at opportunity_created_at stage  \
0       61            1061      2026-01-05             2026-01-06   Won   
1       40            1040      2026-01-03             2026-01-04   Won   
2      223            1223      2026-01-15             2026-01-16  Lost   
3      244            1244      2026-01-17             2026-01-18   Won   
4      305            1305      2026-01-21             2026-01-22   Won   

   amount  conversion_days  
0  3194.0                1  
1  3579.0    

/tmp/ipykernel_54517/2812696461.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  sample_df = pd.read_sql(sample_query, connect)


In [3]:
silver_tables = [
    'SLV_SALES_FUNNEL', 
    'SLV_MARKETING_PERFORMANCE', 
    'SLV_CUSTOMER_JOURNEY'
]

print("--- Starting Connection and Counting ---")

try:
    # 1. فتح الاتصال
    with teradatasql.connect(host=host, user=username, password=password) as connect:
        cursor = connect.cursor()
        print("Connected to Teradata Successfully!")
        
        print("\n--- Bronze Tables Data Counts ---")
        
        # 2. اللف على الجداول وعد الصفوف
        for table in silver_tables:
            try:
                cursor.execute(f"SELECT COUNT(*) FROM {table}")
                row_count = cursor.fetchone()[0]
                print(f"Table {table}: {row_count} rows found.")
            except Exception as table_error:
                print(f"Error checking table {table}: {table_error}")
                
        print("\n-----------------------------------")

except Exception as conn_error:
    print(f"Failed to connect or execute: {conn_error}")

--- Starting Connection and Counting ---
Connected to Teradata Successfully!

--- Bronze Tables Data Counts ---
Table SLV_SALES_FUNNEL: 96544 rows found.
Table SLV_MARKETING_PERFORMANCE: 0 rows found.
Table SLV_CUSTOMER_JOURNEY: 0 rows found.

-----------------------------------


In [ ]:
# كود طوارئ لتصفير الجلسة المعلقة من البايثون
try:
    cursor.execute("""
        UPDATE SNP_SESSION 
        SET SESS_STATUS = 'E', 
            SESS_END = CURRENT_TIMESTAMP 
        WHERE SESS_NO = 8
    """)
    connect.commit()
    print("✅ Session 8 has been forcefully ended.")
except Exception as e:
    print(f"❌ Error: {e}")

In [7]:
tables = ['SLV_MARKETING_PERFORMANCE', 'SLV_CUSTOMER_JOURNEY']
for table in tables:
    try:
        cursor.execute(f"DELETE FROM {table}")
        connect.commit() # مهم جداً في Teradata لتثبيت المسح
        print(f"✅ {table} has been cleared.")
    except Exception as e:
        print(f"⚠️ Could not clear {table}: {e}")

✅ SLV_MARKETING_PERFORMANCE has been cleared.
✅ SLV_CUSTOMER_JOURNEY has been cleared.


In [ ]:
# مسح الاتصال القديم تماماً من الذاكرة
del cursor
del connect

In [2]:
from teradataml import execute_sql

# حذف البيانات القديمة لضمان عدم التكرار (Idempotency)
try:
    execute_sql("DELETE FROM demo_user.SLV_MARKETING_PERFORMANCE;")
    print("Old data cleared.")
except Exception as e:
    print(f"Table might be empty or: {e}")

# تنفيذ الـ Aggregation وتحميل البيانات
# ملحوظة: استخدمنا CAST لأن بيانات البرونز VARCHAR
query = """
INSERT INTO demo_user.SLV_MARKETING_PERFORMANCE (
    campaign_id,
    total_spend,
    total_clicks,
    total_web_events
)
SELECT 
    campaign_id,
    SUM(CAST(spend_amount AS DECIMAL(18,2))) AS total_spend,
    SUM(CAST(clicks AS INTEGER)) AS total_clicks,
    0 AS total_web_events
FROM demo_user.BRZ_ADS_SPEND
GROUP BY 1;
"""

try:
    execute_sql(query)
    print("Success: SLV_MARKETING_PERFORMANCE updated successfully!")
except Exception as e:
    print(f"Error executing query: {e}")

Old data cleared.
Success: SLV_MARKETING_PERFORMANCE updated successfully!


In [5]:
from teradataml import execute_sql

# Step 1: Clear the silver table
execute_sql("DELETE FROM demo_user.SLV_CUSTOMER_JOURNEY")

# Step 2: Perform Identity Resolution and Insert
query = """
INSERT INTO demo_user.SLV_CUSTOMER_JOURNEY (
    fingerprint,
    lead_id,
    event_type,
    event_timestamp
)
SELECT 
    w.cookie_id,
    l.lead_id,
    w.event_name,
    CAST(SUBSTRING(w.event_ts FROM 1 FOR 10) || ' ' || SUBSTRING(w.event_ts FROM 12 FOR 8) AS TIMESTAMP)
FROM demo_user.BRZ_WEB_EVENTS w
JOIN (
    SELECT DISTINCT cookie_id, email_hash 
    FROM demo_user.BRZ_WEB_EVENTS 
    WHERE email_hash IS NOT NULL AND email_hash <> ''
) id_map ON w.cookie_id = id_map.cookie_id
JOIN demo_user.BRZ_CRM_LEADS l ON id_map.email_hash = l.email_hash
"""

try:
    execute_sql(query)
    print("SLV_CUSTOMER_JOURNEY populated successfully.")
except Exception as e:
    print(f"Error: {str(e)}")

SLV_CUSTOMER_JOURNEY populated successfully.


In [6]:
from teradataml import execute_sql
import pandas as pd
import teradatasql
# تنظيف الجدول لضمان دقة البيانات
execute_sql("DELETE FROM demo_user.MAP_CAMPAIGN_ID")

# إضافة كل السيناريوهات المحتملة للربط
mapping_queries = """
INSERT INTO demo_user.MAP_CAMPAIGN_ID (ads_campaign_id, web_utm_campaign)
      SELECT 'CAMP01', 'fb_camp_1'
UNION SELECT 'CAMP01', 'gg_camp_1'
UNION SELECT 'CAMP01', 'em_camp_1'
UNION SELECT 'CAMP02', 'fb_camp_2'
UNION SELECT 'CAMP02', 'gg_camp_2'
UNION SELECT 'CAMP02', 'em_camp_2'
UNION SELECT 'CAMP03', 'org_traffic';
"""

try:
    execute_sql(mapping_queries)
    print("Mapping Table updated with all possible campaign patterns.")
except Exception as e:
    print(f"Error: {e}")

Error: [Version 20.0.0.48] [Session 1507] [Teradata Database] [Error 3888] A SELECT for a UNION,INTERSECT or MINUS must reference a table.
 at gosqldriver/teradatasql.formatError ErrorUtil.go:100
 at gosqldriver/teradatasql.(*teradataConnection).formatDatabaseError ErrorUtil.go:207
 at gosqldriver/teradatasql.(*teradataConnection).makeChainedDatabaseError ErrorUtil.go:223
 at gosqldriver/teradatasql.(*teradataConnection).processErrorParcel TeradataConnection.go:347
 at gosqldriver/teradatasql.(*TeradataRows).processResponseBundle TeradataRows.go:2718
 at gosqldriver/teradatasql.(*TeradataRows).executeSQLRequest TeradataRows.go:1188
 at gosqldriver/teradatasql.newTeradataRows TeradataRows.go:799
 at gosqldriver/teradatasql.(*teradataStatement).QueryContext TeradataStatement.go:122
 at gosqldriver/teradatasql.(*teradataConnection).QueryContext TeradataConnection.go:836
 at database/sql.ctxDriverQuery ctxutil.go:48
 at database/sql.(*DB).queryDC.func1 sql.go:1786
 at database/sql.withLock

In [5]:
from teradataml import execute_sql

# إنشاء جدول المابينج
execute_sql("""
CREATE TABLE demo_user.MAP_CAMPAIGN_ID (
    ads_campaign_id VARCHAR(50),
    web_utm_campaign VARCHAR(50)
) PRIMARY INDEX (ads_campaign_id);
""")

# تغذية الجدول بالقيم الصحيحة (مرة واحدة فقط)
execute_sql("""
INSERT INTO demo_user.MAP_CAMPAIGN_ID VALUES ('CAMP01', 'fb_camp_1');
INSERT INTO demo_user.MAP_CAMPAIGN_ID VALUES ('CAMP02', 'gg_camp_2');
INSERT INTO demo_user.MAP_CAMPAIGN_ID VALUES ('CAMP03', 'fb_camp_3');
""")

TeradataCursor uRowsHandle=10 bClosed=False

In [7]:
update_slv_query = """
REPLACE VIEW demo_user.SLV_MARKETING_PERFORMANCE_V AS
SELECT 
    ads.campaign_id,
    SUM(CAST(ads.spend_amount AS DECIMAL(18,2))) AS total_spend,
    SUM(CAST(ads.clicks AS INTEGER)) AS total_clicks,
    (SELECT COUNT(*) FROM demo_user.BRZ_WEB_EVENTS w 
     JOIN demo_user.MAP_CAMPAIGN_ID m ON w.utm_campaign = m.web_utm_campaign
     WHERE m.ads_campaign_id = ads.campaign_id) AS total_web_events
FROM demo_user.BRZ_ADS_SPEND ads
GROUP BY 1;
"""


In [8]:
from teradataml import DataFrame, execute_sql

# التحقق من وجود حملات في الإعلانات غير موجودة في جدول الجسر
check_mapping_query = """
SELECT DISTINCT a.campaign_id
FROM demo_user.BRZ_ADS_SPEND a
LEFT JOIN demo_user.MAP_CAMPAIGN_ID m ON a.campaign_id = m.ads_campaign_id
WHERE m.ads_campaign_id IS NULL;
"""
res = execute_sql(check_mapping_query)
print("Campaigns missing from Mapping Table:")
print(res.fetchall())

Campaigns missing from Mapping Table:
[['CAMP02'], ['CAMP03'], ['CAMP01']]


In [10]:
from teradataml import execute_sql

# التحقق من وجود حملات إعلانية تائهة (بدون مابينج)
check_mapping = """
SELECT DISTINCT a.campaign_id
FROM demo_user.BRZ_ADS_SPEND a
LEFT JOIN demo_user.MAP_CAMPAIGN_ID m ON a.campaign_id = m.ads_campaign_id
WHERE m.ads_campaign_id IS NULL;
"""
res = execute_sql(check_mapping).fetchall()
if not res:
    print("✅ All Ads Campaigns are mapped correctly!")
else:
    print(f"⚠️ Warning: These campaigns are missing from mapping: {res}")

⚠️ Warning: These campaigns are missing from mapping: [['CAMP02'], ['CAMP03'], ['CAMP01']]


In [11]:
from teradataml import DataFrame
try:
    print(DataFrame("demo_user.MAP_CAMPAIGN_ID"))
except:
    print("Table is either empty or does not exist!")

Table is either empty or does not exist!


In [12]:
from teradataml import execute_sql

# مسح أي بيانات قديمة (لو وجدت)
execute_sql("DELETE FROM demo_user.MAP_CAMPAIGN_ID")

# إضافة كل الاحتمالات اللي اتكلمنا فيها
setup_mapping = """
INSERT INTO demo_user.MAP_CAMPAIGN_ID (ads_campaign_id, web_utm_campaign)
      SELECT 'CAMP01', 'fb_camp_1'
UNION SELECT 'CAMP01', 'gg_camp_1'
UNION SELECT 'CAMP01', 'em_camp_1'
UNION SELECT 'CAMP02', 'fb_camp_2'
UNION SELECT 'CAMP02', 'gg_camp_2'
UNION SELECT 'CAMP02', 'em_camp_2'
UNION SELECT 'CAMP03', 'org_traffic';
"""

try:
    execute_sql(setup_mapping)
    print("✅ Mapping table populated successfully!")
except Exception as e:
    print(f"❌ Error during insert: {e}")

❌ Error during insert: [Version 20.0.0.48] [Session 1507] [Teradata Database] [Error 3888] A SELECT for a UNION,INTERSECT or MINUS must reference a table.
 at gosqldriver/teradatasql.formatError ErrorUtil.go:100
 at gosqldriver/teradatasql.(*teradataConnection).formatDatabaseError ErrorUtil.go:207
 at gosqldriver/teradatasql.(*teradataConnection).makeChainedDatabaseError ErrorUtil.go:223
 at gosqldriver/teradatasql.(*teradataConnection).processErrorParcel TeradataConnection.go:347
 at gosqldriver/teradatasql.(*TeradataRows).processResponseBundle TeradataRows.go:2718
 at gosqldriver/teradatasql.(*TeradataRows).executeSQLRequest TeradataRows.go:1188
 at gosqldriver/teradatasql.newTeradataRows TeradataRows.go:799
 at gosqldriver/teradatasql.(*teradataStatement).QueryContext TeradataStatement.go:122
 at gosqldriver/teradatasql.(*teradataConnection).QueryContext TeradataConnection.go:836
 at database/sql.ctxDriverQuery ctxutil.go:48
 at database/sql.(*DB).queryDC.func1 sql.go:1786
 at datab

In [13]:
from teradataml import execute_sql

# مسح الجدول عشان نبدأ على نظافة
execute_sql("DELETE FROM demo_user.MAP_CAMPAIGN_ID")

# الإدخال بصيغة تيراداتا الصحيحة
# ملحوظة: بنكتب VALUES مرة واحدة وبنحط كل الصفوف بين أقواس
setup_mapping = """
INSERT INTO demo_user.MAP_CAMPAIGN_ID (ads_campaign_id, web_utm_campaign)
VALUES 
    ('CAMP01', 'fb_camp_1'),
    ('CAMP01', 'gg_camp_1'),
    ('CAMP01', 'em_camp_1'),
    ('CAMP02', 'fb_camp_2'),
    ('CAMP02', 'gg_camp_2'),
    ('CAMP02', 'em_camp_2'),
    ('CAMP03', 'org_traffic');
"""

try:
    execute_sql(setup_mapping)
    print("✅ Success! Mapping table is now populated.")
except Exception as e:
    print(f"❌ Still an error: {e}")

❌ Still an error: [Version 20.0.0.48] [Session 1507] [Teradata Database] [Error 3706] Syntax error: expected something between ')' and ','.
 at gosqldriver/teradatasql.formatError ErrorUtil.go:100
 at gosqldriver/teradatasql.(*teradataConnection).formatDatabaseError ErrorUtil.go:207
 at gosqldriver/teradatasql.(*teradataConnection).makeChainedDatabaseError ErrorUtil.go:223
 at gosqldriver/teradatasql.(*teradataConnection).processErrorParcel TeradataConnection.go:347
 at gosqldriver/teradatasql.(*TeradataRows).processResponseBundle TeradataRows.go:2718
 at gosqldriver/teradatasql.(*TeradataRows).executeSQLRequest TeradataRows.go:1188
 at gosqldriver/teradatasql.newTeradataRows TeradataRows.go:799
 at gosqldriver/teradatasql.(*teradataStatement).QueryContext TeradataStatement.go:122
 at gosqldriver/teradatasql.(*teradataConnection).QueryContext TeradataConnection.go:836
 at database/sql.ctxDriverQuery ctxutil.go:48
 at database/sql.(*DB).queryDC.func1 sql.go:1786
 at database/sql.withLoc

In [14]:
from teradataml import execute_sql

# 1. مسح الجدول
execute_sql("DELETE FROM demo_user.MAP_CAMPAIGN_ID")

# 2. الإدخال باستخدام UNION ALL وجدول نظام (لأن تيراداتا تتطلب FROM في الـ SELECT)
setup_mapping = """
INSERT INTO demo_user.MAP_CAMPAIGN_ID (ads_campaign_id, web_utm_campaign)
SELECT 'CAMP01', 'fb_camp_1' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t
UNION ALL SELECT 'CAMP01', 'gg_camp_1' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t
UNION ALL SELECT 'CAMP01', 'em_camp_1' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t
UNION ALL SELECT 'CAMP02', 'fb_camp_2' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t
UNION ALL SELECT 'CAMP02', 'gg_camp_2' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t
UNION ALL SELECT 'CAMP02', 'em_camp_2' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t
UNION ALL SELECT 'CAMP03', 'org_traffic' FROM (SELECT TOP 1 * FROM DBC.TVM) AS t;
"""

try:
    execute_sql(setup_mapping)
    print("✅ Finally! Mapping table is populated.")
except Exception as e:
    # لو الجدول DBC.TVM فيه مشكلة في الصلاحيات، جرب الإدخال الفردي (أضمن حل)
    print("Fallback: Using Individual Inserts...")
    data = [
        ('CAMP01', 'fb_camp_1'), ('CAMP01', 'gg_camp_1'), ('CAMP01', 'em_camp_1'),
        ('CAMP02', 'fb_camp_2'), ('CAMP02', 'gg_camp_2'), ('CAMP02', 'em_camp_2'),
        ('CAMP03', 'org_traffic')
    ]
    for ads_id, utm in data:
        execute_sql(f"INSERT INTO demo_user.MAP_CAMPAIGN_ID VALUES ('{ads_id}', '{utm}')")
    print("✅ Successfully populated using individual inserts.")

✅ Finally! Mapping table is populated.


In [18]:
from teradataml import execute_sql

# الاستعلام المباشر لرؤية النتائج
query = "SELECT TOP 5 * FROM demo_user.SLV_MARKETING_PERFORMANCE"

try:
    results = execute_sql(query).fetchall()
    if results:
        print("✅ Data found in Silver Table:")
        for row in results:
            print(row)
    else:
        print("⚠️ Table exists but it is EMPTY!")
except Exception as e:
    print(f"❌ Database Error: {e}")

✅ Data found in Silver Table:
['CAMP01', Decimal('3495.40'), 7712, 0]
['CAMP03', Decimal('4370.50'), 8603, 0]
['CAMP02', Decimal('2470.70'), 5007, 0]


In [19]:
from teradataml import execute_sql

# كود تصحيح السيلفر: الربط بين الإعلانات والويب عن طريق جدول الجسر
correction_query = """
REPLACE TABLE demo_user.SLV_MARKETING_PERFORMANCE AS
SELECT 
    ads.campaign_id,
    SUM(CAST(ads.spend_amount AS DECIMAL(18,2))) AS total_spend,
    SUM(CAST(ads.clicks AS INTEGER)) AS total_clicks,
    COUNT(DISTINCT web.event_id) AS total_web_events
FROM demo_user.BRZ_ADS_SPEND ads
-- هنا السحر: بنربط الـ ID بالـ UTM عن طريق الجسر
JOIN demo_user.MAP_CAMPAIGN_ID map 
    ON ads.campaign_id = map.ads_campaign_id
-- هنا بنربط بالويب باستخدام الاسم المترجم
LEFT JOIN demo_user.BRZ_WEB_EVENTS web 
    ON map.web_utm_campaign = web.utm_campaign
GROUP BY 1;
"""

try:
    execute_sql(correction_query)
    print("✅ Silver Table has been REBUILT with the Bridge logic!")
except Exception as e:
    print(f"❌ Error during rebuild: {e}")

❌ Error during rebuild: [Version 20.0.0.48] [Session 1507] [Teradata Database] [Error 3707] Syntax error, expected something like a 'METHOD' keyword between the 'REPLACE' keyword and the 'TABLE' keyword.
 at gosqldriver/teradatasql.formatError ErrorUtil.go:100
 at gosqldriver/teradatasql.(*teradataConnection).formatDatabaseError ErrorUtil.go:207
 at gosqldriver/teradatasql.(*teradataConnection).makeChainedDatabaseError ErrorUtil.go:223
 at gosqldriver/teradatasql.(*teradataConnection).processErrorParcel TeradataConnection.go:347
 at gosqldriver/teradatasql.(*TeradataRows).processResponseBundle TeradataRows.go:2718
 at gosqldriver/teradatasql.(*TeradataRows).executeSQLRequest TeradataRows.go:1188
 at gosqldriver/teradatasql.newTeradataRows TeradataRows.go:799
 at gosqldriver/teradatasql.(*teradataStatement).QueryContext TeradataStatement.go:122
 at gosqldriver/teradatasql.(*teradataConnection).QueryContext TeradataConnection.go:836
 at database/sql.ctxDriverQuery ctxutil.go:48
 at datab

In [20]:
from teradataml import execute_sql

try:
    execute_sql("DELETE FROM demo_user.SLV_MARKETING_PERFORMANCE")
    print("✅ Old zero-data cleared.")
except Exception as e:
    print(f"Error clearing table: {e}")

✅ Old zero-data cleared.


In [21]:
correction_insert = """
INSERT INTO demo_user.SLV_MARKETING_PERFORMANCE (
    campaign_id, 
    total_spend, 
    total_clicks, 
    total_web_events
)
SELECT 
    ads.campaign_id,
    SUM(CAST(ads.spend_amount AS DECIMAL(18,2))),
    SUM(CAST(ads.clicks AS INTEGER)),
    COUNT(DISTINCT web.event_id)
FROM demo_user.BRZ_ADS_SPEND ads
-- الربط الأساسي مع الجسر
JOIN demo_user.MAP_CAMPAIGN_ID map 
    ON ads.campaign_id = map.ads_campaign_id
-- الربط مع الويب باستخدام الاسم المترجم
LEFT JOIN demo_user.BRZ_WEB_EVENTS web 
    ON map.web_utm_campaign = web.utm_campaign
GROUP BY 1;
"""

try:
    execute_sql(correction_insert)
    print("🚀 SUCCESS! Silver table is now populated with real events.")
except Exception as e:
    print(f"❌ Re-insert failed: {e}")

❌ Re-insert failed: [Version 20.0.0.48] [Session 1507] [Teradata Database] [Error 3707] Syntax error, expected something like an 'ON' keyword between the word 'MAP_CAMPAIGN_ID' and the 'map' keyword.
 at gosqldriver/teradatasql.formatError ErrorUtil.go:100
 at gosqldriver/teradatasql.(*teradataConnection).formatDatabaseError ErrorUtil.go:207
 at gosqldriver/teradatasql.(*teradataConnection).makeChainedDatabaseError ErrorUtil.go:223
 at gosqldriver/teradatasql.(*teradataConnection).processErrorParcel TeradataConnection.go:347
 at gosqldriver/teradatasql.(*TeradataRows).processResponseBundle TeradataRows.go:2718
 at gosqldriver/teradatasql.(*TeradataRows).executeSQLRequest TeradataRows.go:1188
 at gosqldriver/teradatasql.newTeradataRows TeradataRows.go:799
 at gosqldriver/teradatasql.(*teradataStatement).QueryContext TeradataStatement.go:122
 at gosqldriver/teradatasql.(*teradataConnection).QueryContext TeradataConnection.go:836
 at database/sql.ctxDriverQuery ctxutil.go:48
 at database/

In [22]:
from teradataml import execute_sql

# التأكد من مسح الداتا القديمة مرة تانية للاحتياط
execute_sql("DELETE FROM demo_user.SLV_MARKETING_PERFORMANCE")

# الكود بصيغة واضحة للمحرك
correction_insert = """
INSERT INTO demo_user.SLV_MARKETING_PERFORMANCE (
    campaign_id, 
    total_spend, 
    total_clicks, 
    total_web_events
)
SELECT 
    ads.campaign_id,
    SUM(CAST(ads.spend_amount AS DECIMAL(18,2))),
    SUM(CAST(ads.clicks AS INTEGER)),
    COUNT(DISTINCT web.event_id)
FROM demo_user.BRZ_ADS_SPEND AS ads
INNER JOIN demo_user.MAP_CAMPAIGN_ID AS map 
    ON ads.campaign_id = map.ads_campaign_id
LEFT JOIN demo_user.BRZ_WEB_EVENTS AS web 
    ON map.web_utm_campaign = web.utm_campaign
GROUP BY 1;
"""

try:
    execute_sql(correction_insert)
    print("🚀 BOOM! Silver table is FINALLY populated with real data.")
except Exception as e:
    print(f"❌ Still giving trouble: {e}")

❌ Still giving trouble: [Version 20.0.0.48] [Session 1507] [Teradata Database] [Error 3707] Syntax error, expected something like a name or a Unicode delimited identifier between the 'AS' keyword and the 'map' keyword.
 at gosqldriver/teradatasql.formatError ErrorUtil.go:100
 at gosqldriver/teradatasql.(*teradataConnection).formatDatabaseError ErrorUtil.go:207
 at gosqldriver/teradatasql.(*teradataConnection).makeChainedDatabaseError ErrorUtil.go:223
 at gosqldriver/teradatasql.(*teradataConnection).processErrorParcel TeradataConnection.go:347
 at gosqldriver/teradatasql.(*TeradataRows).processResponseBundle TeradataRows.go:2718
 at gosqldriver/teradatasql.(*TeradataRows).executeSQLRequest TeradataRows.go:1188
 at gosqldriver/teradatasql.newTeradataRows TeradataRows.go:799
 at gosqldriver/teradatasql.(*teradataStatement).QueryContext TeradataStatement.go:122
 at gosqldriver/teradatasql.(*teradataConnection).QueryContext TeradataConnection.go:836
 at database/sql.ctxDriverQuery ctxutil.

In [23]:
from teradataml import execute_sql

# 1. تنظيف الجدول (إحنا عملنا ده بس للاحتياط)
execute_sql("DELETE FROM demo_user.SLV_MARKETING_PERFORMANCE")

# 2. الإدخال المباشر بدون استخدام AS أو كلمات مستعارة
correction_insert = """
INSERT INTO demo_user.SLV_MARKETING_PERFORMANCE (
    campaign_id, 
    total_spend, 
    total_clicks, 
    total_web_events
)
SELECT 
    demo_user.BRZ_ADS_SPEND.campaign_id,
    SUM(CAST(demo_user.BRZ_ADS_SPEND.spend_amount AS DECIMAL(18,2))),
    SUM(CAST(demo_user.BRZ_ADS_SPEND.clicks AS INTEGER)),
    COUNT(DISTINCT demo_user.BRZ_WEB_EVENTS.event_id)
FROM demo_user.BRZ_ADS_SPEND
INNER JOIN demo_user.MAP_CAMPAIGN_ID 
    ON demo_user.BRZ_ADS_SPEND.campaign_id = demo_user.MAP_CAMPAIGN_ID.ads_campaign_id
LEFT JOIN demo_user.BRZ_WEB_EVENTS 
    ON demo_user.MAP_CAMPAIGN_ID.web_utm_campaign = demo_user.BRZ_WEB_EVENTS.utm_campaign
GROUP BY 1;
"""

try:
    execute_sql(correction_insert)
    print("🚀 FINALLY! The data is in.")
except Exception as e:
    print(f"❌ Error: {e}")

🚀 FINALLY! The data is in.


In [2]:
from teradataml import execute_sql

# --- خطوة 1: تنظيف المكان ---
for tbl in ["DIM_MARKETING_CAMPAIGNS", "DIM_LEADS_JOURNEY", "FACT_MARKETING_ANALYTICS"]:
    try: execute_sql(f"DROP TABLE demo_user.{tbl}")
    except: pass

# --- خطوة 2: إنشاء جدول الأبعاد التسويقية الشامل ---
execute_sql("""
CREATE TABLE demo_user.DIM_MARKETING_CAMPAIGNS AS (
    SELECT 
        ads_campaign_id AS campaign_id,
        CASE 
            WHEN web_utm_campaign LIKE 'fb%' THEN 'Facebook'
            WHEN web_utm_campaign LIKE 'gg%' THEN 'Google'
            WHEN web_utm_campaign LIKE 'em%' THEN 'Email'
            ELSE 'Organic' 
        END AS platform,
        CASE 
            WHEN web_utm_campaign LIKE '%paid%' THEN 'Paid Search'
            WHEN web_utm_campaign LIKE '%social%' THEN 'Social Media'
            ELSE 'Direct/Referral'
        END AS channel_group
    FROM demo_user.MAP_CAMPAIGN_ID
) WITH DATA PRIMARY INDEX (campaign_id);
""")

# --- خطوة 3: إنشاء جدول الحقائق (The Big Fact Table) ---
# الجدول ده هيربط الـ Silver ببعضه عشان يطلع أرقام الـ Dashboard
execute_sql("""
CREATE TABLE demo_user.FACT_MARKETING_ANALYTICS AS (
    SELECT 
        s.campaign_id,
        j.lead_id,
        s.total_spend,
        s.total_web_events,
        COALESCE(f.amount, 0) AS revenue,
        COALESCE(f.conversion_days, 0) AS days_to_convert,
        CASE WHEN f.amount > 0 THEN 1 ELSE 0 END AS is_sale
    FROM demo_user.SLV_MARKETING_PERFORMANCE s
    LEFT JOIN demo_user.MAP_CAMPAIGN_ID m ON s.campaign_id = m.ads_campaign_id
    LEFT JOIN demo_user.SLV_CUSTOMER_JOURNEY j ON m.web_utm_campaign = j.event_type
    LEFT JOIN demo_user.SLV_SALES_FUNNEL f ON j.lead_id = f.lead_id
) WITH DATA PRIMARY INDEX (campaign_id, lead_id);
""")

print("🎯 GOLD LAYER REBUILT: Now you have a robust Star Schema for a 2-page Dashboard!")

🎯 GOLD LAYER REBUILT: Now you have a robust Star Schema for a 2-page Dashboard!


In [3]:
from teradataml import execute_sql

# قائمة بجداول الـ Gold اللي كريتناها في الخطوة اللي فاتت
gold_tables = [
    "DIM_MARKETING_CAMPAIGNS",
    "FACT_MARKETING_ANALYTICS"
]

print("📊 --- Gold Layer Row Count Check --- 📊\n")

for table in gold_tables:
    query = f"SELECT COUNT(*) FROM demo_user.{table}"
    try:
        # تنفيذ الكود وجلب النتيجة
        count = execute_sql(query).fetchone()[0]
        
        status = "✅" if count > 0 else "⚠️ EMPTY"
        print(f"{status} Table: {table.ljust(25)} | Rows: {count}")
        
    except Exception as e:
        print(f"❌ Error reading {table}: {e}")

print("\n" + "="*40)

📊 --- Gold Layer Row Count Check --- 📊

✅ Table: DIM_MARKETING_CAMPAIGNS   | Rows: 7
✅ Table: FACT_MARKETING_ANALYTICS  | Rows: 3



In [6]:
execute_sql("""
CREATE TABLE demo_user.DIM_LEADS_JOURNEY (
    lead_id INTEGER,
    first_touch_point VARCHAR(50),
    lead_status VARCHAR(50)
) PRIMARY INDEX (lead_id);
""")

TeradataCursor uRowsHandle=18 bClosed=False

In [7]:
execute_sql("""
INSERT INTO demo_user.DIM_LEADS_JOURNEY
SELECT 
    j.lead_id,
    j.event_type,
    CASE WHEN s.lead_id IS NOT NULL THEN 'Converted' ELSE 'Potential' END
FROM demo_user.SLV_CUSTOMER_JOURNEY j
LEFT JOIN demo_user.SLV_SALES_FUNNEL s ON j.lead_id = s.lead_id;
""")

TeradataCursor uRowsHandle=19 bClosed=False

In [8]:
gold_tables = ["DIM_MARKETING_CAMPAIGNS", "DIM_LEADS_JOURNEY", "FACT_MARKETING_ANALYTICS"]

print("\n📊 --- Row Count Validation --- 📊")
for table in gold_tables:
    count = execute_sql(f"SELECT COUNT(*) FROM demo_user.{table}").fetchone()[0]
    print(f"Table: {table.ljust(25)} | Rows: {count}")


📊 --- Row Count Validation --- 📊
Table: DIM_MARKETING_CAMPAIGNS   | Rows: 7
Table: DIM_LEADS_JOURNEY         | Rows: 862
Table: FACT_MARKETING_ANALYTICS  | Rows: 3


In [10]:
import pandas as pd
from teradataml import DataFrame

# 1. سحب البيانات من Teradata إلى Pandas DataFrames
df_campaigns = DataFrame("DIM_MARKETING_CAMPAIGNS").to_pandas()
df_leads = DataFrame("DIM_LEADS_JOURNEY").to_pandas()
df_fact = DataFrame("FACT_MARKETING_ANALYTICS").to_pandas()

# 2. تصدير البيانات لملف Excel واحد بـ 3 صفحات
file_name = "Teradata_Gold_Layer_Export.xlsx"

with pd.ExcelWriter(file_name) as writer:
    df_campaigns.to_excel(writer, sheet_name='Dim_Campaigns', index=False)
    df_leads.to_excel(writer, sheet_name='Dim_Leads', index=False)
    df_fact.to_excel(writer, sheet_name='Fact_Performance', index=False)

print(f"✅ تم بنجاح! الملف جاهز باسم: {file_name}")
print("تقدر دلوقتي تنزله من الـ Jupyter Notebook وتشتغل Offline.")

✅ تم بنجاح! الملف جاهز باسم: Teradata_Gold_Layer_Export.xlsx
تقدر دلوقتي تنزله من الـ Jupyter Notebook وتشتغل Offline.


In [13]:
from teradataml import execute_sql

def safe_drop(table_name):
    try:
        execute_sql(f"DROP TABLE {table_name}")
    except:
        pass

# A. تنظيف Ads Spend (استخدام BRZ_ADS_SPEND)
safe_drop("demo_user.SLV_ADS_SPEND")
execute_sql("""
CREATE TABLE demo_user.SLV_ADS_SPEND AS (
    SELECT 
        spend_date, 
        CASE 
            WHEN LOWER(source_system) = 'meta' THEN 'facebook'
            ELSE LOWER(source_system) 
        END as platform, 
        CASE 
            WHEN campaign_id IN ('fb_camp_1', 'gg_camp_1') THEN 'CAMP01'
            WHEN campaign_id IN ('fb_camp_2', 'gg_camp_2') THEN 'CAMP02'
            ELSE campaign_id 
        END as campaign_id_clean,
        impressions,
        clicks, 
        spend_amount
    FROM demo_user.BRZ_ADS_SPEND
) WITH DATA PRIMARY INDEX (spend_date, campaign_id_clean);
""")

# B. تنظيف CRM Leads (استخدام BRZ_CRM_LEADS)
safe_drop("demo_user.SLV_CRM_LEADS")
execute_sql("""
CREATE TABLE demo_user.SLV_CRM_LEADS AS (
    SELECT 
        lead_id, created_at, email_hash, 
        LOWER(lead_source) as lead_source_clean
    FROM demo_user.BRZ_CRM_LEADS
) WITH DATA PRIMARY INDEX (lead_id);
""")

# C. تنظيف Web Events (استخدام BRZ_WEB_EVENTS)
safe_drop("demo_user.SLV_WEB_EVENTS")
execute_sql("""
CREATE TABLE demo_user.SLV_WEB_EVENTS AS (
    SELECT 
        event_id, event_ts, event_name, cookie_id, email_hash, device_type, geo_city,
        LOWER(utm_source) as utm_source,
        CASE 
            WHEN utm_campaign IN ('fb_camp_1', 'gg_camp_1') THEN 'CAMP01'
            WHEN utm_campaign IN ('fb_camp_2', 'gg_camp_2') THEN 'CAMP02'
            WHEN utm_campaign = 'organic' THEN 'organic'
            ELSE utm_campaign 
        END as utm_campaign_clean
    FROM demo_user.BRZ_WEB_EVENTS
) WITH DATA PRIMARY INDEX (event_id);
""")

print("✅ Silver Layer is now CREATED successfully using the correct Bronze names!")

✅ Silver Layer is now CREATED successfully using the correct Bronze names!


In [15]:
from teradataml import execute_sql

# وظيفة المسح الآمن
def safe_drop(table_name):
    try:
        execute_sql(f"DROP TABLE {table_name}")
    except:
        pass

# إنشاء جدول الـ Gold (Leads + Opportunities فقط)
safe_drop("demo_user.GLD_FINAL_MARKETING_SALES")

execute_sql("""
CREATE TABLE demo_user.GLD_FINAL_MARKETING_SALES AS (
    SELECT 
        -- بيانات الـ Lead من الـ Silver
        l.lead_id,
        l.lead_source_clean,
        l.created_at as lead_created_at,
        l.email_hash,
        
        -- سحب أعمدة الـ Opportunity من البرونز مع استبعاد الـ lead_id المتكرر
        o.opportunity_id,
        o.created_at as opp_created_at,
        o.updated_at as opp_updated_at,
        o.stage,
        o.amount,
        o.currency,
        o.close_date
        
    FROM demo_user.SLV_CRM_LEADS l
    LEFT JOIN demo_user.BRZ_CRM_OPPORTUNITIES o ON l.lead_id = o.lead_id
) WITH DATA PRIMARY INDEX (lead_id);
""")

print("🚀 Gold Layer Built Successfully: Leads & Opportunities merged without ambiguity!")

🚀 Gold Layer Built Successfully: Leads & Opportunities merged without ambiguity!


In [17]:
from teradataml import execute_sql

# وظيفة المسح الآمن
def safe_drop(table_name):
    try:
        execute_sql(f"DROP TABLE {table_name}")
    except:
        pass

safe_drop("demo_user.SLV_WEB_EVENTS")

execute_sql("""
CREATE TABLE demo_user.SLV_WEB_EVENTS AS (
    SELECT 
        event_id, 
        event_ts, 
        event_name, 
        cookie_id, 
        email_hash, 
        device_type, 
        geo_city,
        LOWER(TRIM(utm_source)) as utm_source,
        CASE 
            -- توحيد حملة 1 (Facebook, Google, Email)
            WHEN LOWER(TRIM(utm_campaign)) IN ('fb_camp_1', 'gg_camp_1', 'em_camp_1', 'camp01') THEN 'CAMP01'
            
            -- توحيد حملة 2
            WHEN LOWER(TRIM(utm_campaign)) IN ('fb_camp_2', 'gg_camp_2', 'camp02') THEN 'CAMP02'
            
            -- توحيد الأورجانيك (بما فيها org_traffic اللي ظهرت في الصورة)
            WHEN LOWER(TRIM(utm_campaign)) IN ('organic', 'org_traffic') THEN 'organic'
            
            -- أي حاجة تانية خليها lowercase
            ELSE LOWER(TRIM(utm_campaign)) 
        END as utm_campaign_clean
    FROM demo_user.BRZ_WEB_EVENTS
) WITH DATA PRIMARY INDEX (event_id);
""")

print("✅ SLV_WEB_EVENTS fixed: em_camp_1 -> CAMP01 and org_traffic -> organic")

✅ SLV_WEB_EVENTS fixed: em_camp_1 -> CAMP01 and org_traffic -> organic


In [18]:
from teradataml import execute_sql

# كود عد الصفوف في جدول Silver Web Events
result = execute_sql("SELECT COUNT(*) FROM demo_user.SLV_WEB_EVENTS").fetchone()

print(f"📊 إجمالي عدد الصفوف (Rows) في جدول الـ Silver هو: {result[0]:,}")

# وعشان نتطمن أكتر، ده كود يوريك الـ Campaigns بعد التنظيف وعدد كل واحدة
print("\n🔍 تفاصيل الحملات بعد التنظيف:")
check_dist = execute_sql("""
    SELECT utm_campaign_clean, COUNT(*) as row_count 
    FROM demo_user.SLV_WEB_EVENTS 
    GROUP BY 1 
    ORDER BY 2 DESC
""").fetchall()

for row in check_dist:
    print(f"- {row[0]}: {row[1]:,} rows")

📊 إجمالي عدد الصفوف (Rows) في جدول الـ Silver هو: 2,102

🔍 تفاصيل الحملات بعد التنظيف:
- CAMP01: 1,033 rows
- organic: 557 rows
- CAMP02: 512 rows
